# MPM Pilot — Yilgarn Orogenic Gold
## 01 · Data Audit & Feasibility Check

This notebook audits all required input datasets **before any modelling begins**.
It checks:
1. That all raw files are present in the expected locations
2. Native resolution, CRS, and extent of each dataset
3. Coverage overlap — every input must cover the pilot AOI
4. Label statistics — P / RN / U counts, spatial distribution, coordinate precision
5. Identifies any data gaps that must be resolved before harmonization

> **Guardrail:** This notebook is read-only for data. It produces a `data_audit_report.md` in `outputs/reports/` and raises a `DataAuditError` if any critical gap is found.

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

CONFIG_PATH = '../config/pilot.yaml'
with open(CONFIG_PATH) as f:
    CONFIG = yaml.safe_load(f)

DATA_DIR = Path('../data/raw')
REPORT_DIR = Path('../outputs/reports')
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Pilot: {CONFIG['pilot']['name']}")
print(f"Commodity: {CONFIG['pilot']['commodity']} | Deposit type: {CONFIG['pilot']['deposit_type']}")
print(f"Grid CRS: {CONFIG['grid']['crs']} | Cell size: {CONFIG['grid']['cell_size_m']} m")

## 1. File Presence Check

In [ ]:
EXPECTED_FILES = {
    'aster':         DATA_DIR / 'aster',
    'sentinel2':     DATA_DIR / 'sentinel2',
    'magnetics_tmi': DATA_DIR / 'magnetics' / 'tmi.tif',
    'magnetics_rtp': DATA_DIR / 'magnetics' / 'rtp.tif',
    'radiometrics_k': DATA_DIR / 'radiometrics' / 'potassium.tif',
    'radiometrics_u': DATA_DIR / 'radiometrics' / 'uranium.tif',
    'radiometrics_th': DATA_DIR / 'radiometrics' / 'thorium.tif',
    'gravity':       DATA_DIR / 'gravity' / 'bouguer.tif',
    'dem':           DATA_DIR / 'dem',
    'geology':       DATA_DIR / 'geology',
    'minedex':       DATA_DIR / 'labels' / 'minedex',
    'wamex':         DATA_DIR / 'labels' / 'wamex',
}

audit_rows = []
for name, path in EXPECTED_FILES.items():
    exists = path.exists()
    n_files = len(list(path.iterdir())) if exists and path.is_dir() else (1 if exists else 0)
    status = '✅' if exists else '❌ MISSING'
    audit_rows.append({'dataset': name, 'path': str(path), 'present': exists,
                        'n_files': n_files, 'status': status})
    print(f"{status:4s}  {name:25s}  {str(path)}")

audit_df = pd.DataFrame(audit_rows)
missing = audit_df[~audit_df['present']]
if not missing.empty:
    print(f"\n⚠️  {len(missing)} dataset(s) MISSING — see README.md for download instructions.")
    print("Pipeline will not proceed until all critical datasets are present.")
else:
    print("\n✅ All expected datasets found.")

## 2. Raster Resolution & CRS Audit

In [ ]:
def audit_raster(path: Path) -> dict:
    """Read basic raster metadata without loading pixel data."""
    if not path.exists():
        return {'status': 'MISSING'}
    with rasterio.open(path) as src:
        return {
            'crs':        str(src.crs),
            'res_x_m':    abs(src.transform.a),
            'res_y_m':    abs(src.transform.e),
            'width_px':   src.width,
            'height_px':  src.height,
            'bands':      src.count,
            'dtype':      src.dtypes[0],
            'nodata':     src.nodata,
            'bounds':     src.bounds,
            'status':     'OK',
        }

raster_files = {
    'TMI':        DATA_DIR / 'magnetics' / 'tmi.tif',
    'RTP':        DATA_DIR / 'magnetics' / 'rtp.tif',
    'Potassium':  DATA_DIR / 'radiometrics' / 'potassium.tif',
    'Uranium':    DATA_DIR / 'radiometrics' / 'uranium.tif',
    'Thorium':    DATA_DIR / 'radiometrics' / 'thorium.tif',
    'Gravity':    DATA_DIR / 'gravity' / 'bouguer.tif',
}

raster_audit = {name: audit_raster(p) for name, p in raster_files.items()}
raster_df = pd.DataFrame(raster_audit).T

# Flag any dataset whose native resolution > 200 m
TARGET_CELL_M = CONFIG['grid']['cell_size_m']
print(f"Target grid: {TARGET_CELL_M} m\n")
print(raster_df[['crs', 'res_x_m', 'res_y_m', 'width_px', 'height_px', 'status']].to_string())
print()
for name, row in raster_df.iterrows():
    if row.get('status') == 'OK':
        res = float(row['res_x_m'])
        if res > 400:
            print(f"⚠️  {name}: {res:.0f} m native — will be upsampled {res/TARGET_CELL_M:.0f}× → MARK WITH INTERPOLATION FLAG")
        elif res > TARGET_CELL_M:
            print(f"ℹ️  {name}: {res:.0f} m native — upsampled {res/TARGET_CELL_M:.1f}× to {TARGET_CELL_M} m (acceptable)")
        else:
            print(f"✅  {name}: {res:.0f} m native — native resolution finer than target cell")

## 3. Label Coordinate Precision Check

In [ ]:
minedex_dir = DATA_DIR / 'labels' / 'minedex'
if not minedex_dir.exists() or not list(minedex_dir.iterdir()):
    print("❌ MINEDEX data not yet downloaded. Skipping label audit.")
else:
    # Load first available file
    label_file = next(minedex_dir.iterdir())
    if label_file.suffix == '.csv':
        labels = pd.read_csv(label_file)
    else:
        labels = gpd.read_file(label_file)

    print(f"MINEDEX records: {len(labels)}")
    print(labels.dtypes)

    # Coordinate precision: check if lat/lon are suspiciously rounded
    if 'latitude' in labels.columns and 'longitude' in labels.columns:
        lat_decimals = labels['latitude'].dropna().apply(lambda x: len(str(x).split('.')[-1]) if '.' in str(x) else 0)
        print(f"\nLatitude decimal places (min/median/max): "
              f"{lat_decimals.min()} / {lat_decimals.median()} / {lat_decimals.max()}")
        precision_ok = lat_decimals >= 4   # < 4 decimal places → ~10 m precision at best
        print(f"Records with ≥4 decimal places (≈11 m precision): {precision_ok.sum()} / {len(labels)}")
        if precision_ok.mean() < 0.5:
            print("⚠️  >50% of records have <4 decimal-place precision — verify coordinate epoch and source")

## 4. AOI Coverage Check

In [ ]:
from shapely.geometry import box

aoi_bounds = CONFIG['grid']['aoi_bounds_geo']  # [W, S, E, N]
aoi_poly = box(*aoi_bounds)
aoi_gdf = gpd.GeoDataFrame({'geometry': [aoi_poly]}, crs='EPSG:4326')

fig, ax = plt.subplots(figsize=(8, 6))
aoi_gdf.plot(ax=ax, edgecolor='red', facecolor='none', linewidth=2, label='Pilot AOI')
ax.set_title('Pilot AOI — Eastern Goldfields, Yilgarn Craton')
ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
ax.legend()

# Overlay raster extents if available
for name, info in raster_audit.items():
    if info.get('status') == 'OK':
        b = info['bounds']
        raster_box = box(b.left, b.bottom, b.right, b.top)
        raster_gdf = gpd.GeoDataFrame({'geometry': [raster_box]}, crs=info['crs']).to_crs('EPSG:4326')
        raster_gdf.plot(ax=ax, edgecolor='blue', facecolor='none', linestyle='--', alpha=0.5, label=name)

plt.tight_layout()
plt.savefig(str(REPORT_DIR / 'aoi_coverage_check.png'), dpi=150)
plt.show()
print(f"AOI bounds (WGS84): W={aoi_bounds[0]}, S={aoi_bounds[1]}, E={aoi_bounds[2]}, N={aoi_bounds[3]}")
area_approx_km2 = (aoi_bounds[2]-aoi_bounds[0]) * 111.32 * (aoi_bounds[3]-aoi_bounds[1]) * 110.57
n_cells = area_approx_km2 * 1e6 / (TARGET_CELL_M**2)
print(f"Approximate AOI area: {area_approx_km2:,.0f} km²")
print(f"Approximate grid cells at {TARGET_CELL_M} m: {n_cells:,.0f}")

## 5. Audit Summary Report

In [ ]:
report_lines = [
    f"# Data Audit Report — {CONFIG['pilot']['name']}",
    f"Generated: {pd.Timestamp.now().isoformat()}",
    "",
    "## File Presence",
    audit_df[['dataset','present','status']].to_markdown(index=False),
    "",
    "## Raster Metadata",
    raster_df[['crs','res_x_m','res_y_m','width_px','height_px','bands','dtype','status']].to_markdown(),
    "",
    "## Critical Actions Required Before Harmonization",
]

missing_list = audit_df[~audit_df['present']]['dataset'].tolist()
if missing_list:
    for m in missing_list:
        report_lines.append(f"- [ ] Download `{m}` — see README.md for instructions")
else:
    report_lines.append("- All datasets present. Proceed to harmonization stage.")

report_path = REPORT_DIR / 'data_audit_report.md'
report_path.write_text('\n'.join(report_lines))
print(f"Audit report written to: {report_path}")